<a href="https://colab.research.google.com/github/egonzalez100522276/p1-aprendizaje-automatico-100522276-100522289/blob/main/1_eda_eleccion_modelos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Práctica 1. Aprendizaje Automático

## Grado en Ingeniería Informática - UC3M

- Ernesto González Cerezo: 100522276@alumnos.uc3m.es
- David Benito Gil: 100522289@alumnos.uc3m.es



## 1. Introducción y Objetivos


Esta práctica trata sobre la creación y entrenamiento de un modelo predictivo orientado al mundo financiero, más concretamente, a los préstamos de un banco.

Se quiere predecir, a partir de una serie de datos de un cliente, si éste aceptará el producto o no.

## 2. EDA simplificado


El primer paso para realizar el análisis explorativo de datos (EDA) es determinar cuántas variables e instancias hay en el conjunto de datos.

### Carga del dataset desde GitHub

In [2]:
!rm -rf p1-aprendizaje-automatico-100522276-100522289
!git clone https://github.com/egonzalez100522276/p1-aprendizaje-automatico-100522276-100522289.git

Cloning into 'p1-aprendizaje-automatico-100522276-100522289'...
remote: Enumerating objects: 112, done.
remote: Counting objects: 100% (112/112), done.
remote: Compressing objects: 100% (15/15), done.
remote: Total 112 (delta 98), reused 103 (delta 96), pack-reused 0 (from 0)
Receiving objects: 100% (112/112), 958.57 KiB | 13.69 MiB/s, done.
Resolving deltas: 100% (98/98), done.


In [3]:
import pandas as pd
import numpy as np

# Definir la ruta al dataset
DATASET_PATH = "p1-aprendizaje-automatico-100522276-100522289/bank_ALL/bank_17.pkl"

# Cambia el nombre por tu fichero real
df = pd.read_pickle(DATASET_PATH)

print("Shape (filas, columnas):", df.shape)
#display(df.head(3))
# Fijar la semilla 'random_state' para garantizar la reproducibilidad
display(df.sample(10, random_state=42))

Shape (filas, columnas): (11000, 17)


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,deposit
109,41,blue-collar,married,primary,no,1250,yes,no,unknown,20,may,1392,2,-1,0,unknown,yes
5555,36,technician,married,secondary,no,0,yes,no,cellular,21,jul,193,5,-1,0,unknown,no
7093,45,services,married,unknown,no,8319,no,no,cellular,15,jun,75,2,-1,0,unknown,no
4028,32,student,single,secondary,no,1102,no,no,telephone,8,oct,194,1,-1,0,unknown,yes
3146,68,retired,married,tertiary,no,2812,no,no,cellular,3,feb,279,2,181,1,failure,yes
4085,46,technician,married,unknown,no,3308,no,no,cellular,27,oct,171,1,91,2,success,yes
3046,25,technician,married,tertiary,no,2551,no,no,cellular,28,dec,400,3,-1,0,unknown,yes
6696,30,admin.,married,secondary,no,350,no,yes,cellular,9,jul,171,5,-1,0,unknown,no
4266,35,management,single,tertiary,no,7918,no,no,cellular,7,sep,497,1,-1,0,unknown,yes
8878,45,blue-collar,married,secondary,no,-221,yes,yes,unknown,3,jun,170,1,-1,0,unknown,no


### Análisis de las variables


#### Identificación de variables categóricas

In [4]:
import pandas as pd

# Columnas categóricas
cat_cols = df.select_dtypes(include=["object"]).columns

resumen = []

# Obtener cardinalidades de cada variable categórica
for col in cat_cols:
    cardinalidad = df[col].nunique()
    valores = df[col].dropna().unique().tolist()
    alta_card = cardinalidad > 10   # umbral arbitrario

    # Añadir columnas a la tabla
    resumen.append({
        "Variable": col,
        "Cardinalidad": cardinalidad,
        "Valores posibles": valores,
        "Alta cardinalidad (> 10)": alta_card
    })

# Construir la tabla
tabla_resumen = pd.DataFrame(resumen)

# Mostrar tabla
tabla_resumen

,Variable,Cardinalidad,Valores posibles,Alta cardinalidad (> 10)
0,job,12,"[admin., technician, services, management, ret...",True
1,marital,3,"[married, single, divorced]",False
2,education,4,"[secondary, tertiary, primary, unknown]",False
3,default,2,"[no, yes]",False
4,housing,2,"[yes, no]",False
5,loan,2,"[no, yes]",False
6,contact,3,"[unknown, cellular, telephone]",False
7,month,12,"[may, jun, jul, aug, oct, nov, dec, jan, feb, ...",True
8,poutcome,4,"[unknown, other, failure, success]",False
9,deposit,2,"[yes, no]",False


#### Missing Values /NA's

In [5]:
# Identificar variables con missing values
import numpy as np

resumen = []

for col in df.columns:
    # En pdays, los missing values se codifican con "-1"
    if col == "pdays":
        missing = df[col].isna().sum() + (df[col] == -1).sum()

    # El valor 'unknown' se computa como missing value
    else:
        missing = df[col].isna().sum() + (df[col] == "unknown").sum()
    # Añadir columnas a la tabla
    resumen.append({
        "Variable": col,
        "Missing totales": missing,
        "% Missing": (missing / len(df)) * 100
    })
# Construir la tabla
tabla_missing = pd.DataFrame(resumen).sort_values("% Missing", ascending=False)

# Mostrar tabla
tabla_missing

,Variable,Missing totales,% Missing
15,poutcome,8205,74.590909
13,pdays,8203,74.572727
8,contact,2309,20.990909
3,education,860,7.818182
1,job,68,0.618182
0,age,0,0.000000
5,balance,0,0.000000
4,default,0,0.000000
2,marital,0,0.000000
7,loan,0,0.000000


La variable *pdays* indica el número de días transcurridos desde el último contacto con el cliente en una campaña anterior.

En este dataset el valor -1 indica que el cliente no fue contactado previamente, por lo que no representa un valor perdido real sino una codificación especial.

Por otro lado, la variable *poutcome* presenta un alto porcentaje de valores unknown, lo cual indica que en muchos casos no existe información sobre el resultado de campañas anteriores.

#### Columnas constantes y columnas tipo ID

In [6]:
# Construir tabla de columnas constantes
tabla_constantes = pd.DataFrame({
    "Variable": df.columns,
    "Valores únicos": df.nunique(dropna=False)
})

tabla_constantes = tabla_constantes[tabla_constantes["Valores únicos"] == 1]

tabla_constantes

,Variable,Valores únicos


In [7]:
# Construir tabla de columnas tipo ID
tabla_id = pd.DataFrame({
    "Variable": df.columns,
    "Cardinalidad": df.nunique(),
})

tabla_id["% unicidad"] = tabla_id["Cardinalidad"] / len(df)
tabla_id["¿Es columna ID?"] = tabla_id["% unicidad"] > 0.95  # umbral arbitrario

tabla_id = tabla_id.sort_values("% unicidad", ascending=False)

tabla_id
tabla_id[tabla_id["¿Es columna ID?"]]

,Variable,Cardinalidad,% unicidad,¿Es columna ID?


Se ha analizado si existen columnas que puedan actuar como identificadores únicos de las instancias. Para ello se ha calculado el porcentaje de unicidad de cada variable respecto al número total de instancias.

No se han encontrado variables con un porcentaje de unicidad cercano al 100%, por lo que se concluye que no existen columnas tipo ID en el dataset.

#### Tipo de problema

In [8]:
df["deposit"].value_counts()

,count
deposit,
no,5780
yes,5220


Viendo este resultado se puede ver claramente un tipo de problema de clasificación y en este caso binaria (yes, no)

#### Desbalanceo

In [9]:
df["deposit"].value_counts(normalize=True) * 100

,proportion
deposit,
no,52.545455
yes,47.454545


Viendo que las clases están más o menos cerca, se podría decir que no está fuertemente desbalanceado.

# 3. Método de Evaluación

## Elección de métrica de evaluación

A la hora de elegir la métrica más adecuada, es fundamental mirar ante qué tipo de problema nos encontramos: **clasificación binaria**.

#### Accuracy
Inicialmlente, se nos vino a la cabeza la métrica de **Accuracy**, pero nos dimos cuenta de que tiene un problema fundamental al aplicarla a clasificación binaria, y es que si la proporción entre clases está descompensada y una clase es muy mayoritaria, un modelo naive que siempre prediga la clase predominante tendrá una gran accuracy, siendo ciego al resto de variables y sin capturar patrones en el dataset.

#### F1-score
Esta métrica combina *precision* y *recall*, guardando un buen equilibrio entre **no perder potenciales clientes** (gracias al *recall*), y **no contactar a clientes potencialmente no interesados** (gracias a *precision*).

Por tanto, **elegiremos F1-Score** como métrica de evaluación para el modelo durante la práctica.

## Holdout
El split elegido es **Holdout**, dedicando 2/3 del dataset a *train*, y 1/3 a *test*.

In [10]:
from sklearn.model_selection import train_test_split

# Definimos nuestros conjuntos X e Y
X = df.drop("deposit", axis=1)
y = df["deposit"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=1/3,   # 1/3 test, 2/3 train
    random_state=42,  # reproducibilidad
    stratify=y
)

# Comprobaciones
print("------------------------------")
print("Tamaño de train:", len(X_train))
print("Tamaño de test:", len(X_test))
print("------------------------------")
print("Forma de X_train: ", X_train.shape)
print("Forma de X_test: ", X_test.shape)
print("------------------------------")
print("Proporción de clases en train:")
print(y_train.value_counts(normalize=True) * 100)
print("\n")
print("Proporción de clases en test:")
print(y_test.value_counts(normalize=True) * 100)

------------------------------
Tamaño de train: 7333
Tamaño de test: 3667
------------------------------
Forma de X_train:  (7333, 16)
Forma de X_test:  (3667, 16)
------------------------------
Proporción de clases en train:
deposit
no     52.543297
yes    47.456703
Name: proportion, dtype: float64


Proporción de clases en test:
deposit
no     52.549768
yes    47.450232
Name: proportion, dtype: float64


# 4. Preprocesado de los datos

Antes de entrenar los modelos, es necesario aplicar un preprocesado a los datos.

En las variables numéricas se imputarán los posibles valores faltantes con la media.  
En las variables categóricas se imputarán con el valor más frecuente y después se aplicará una codificación One-Hot Encoding para poder utilizarlas en los modelos.

In [11]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# Identificación de variables numéricas y categóricas
num_cols = X_train.select_dtypes(include=["int64", "float64"]).columns
cat_cols = X_train.select_dtypes(include=["object"]).columns

print("Variables numéricas:", list(num_cols))
print("Variables categóricas:", list(cat_cols))

Variables numéricas: ['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous']
Variables categóricas: ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']


Se observa que el conjunto de entrenamiento contiene 7 variables numéricas y 9 variables categóricas.  
Esto hace necesario aplicar un preprocesado diferente a cada tipo de variable.

## Construcción del preprocesador
Se construye un preprocesador mediante `ColumnTransformer`, aplicando imputación por media a las variables numéricas e imputación por moda junto con One-Hot Encoding a las variables categóricas.

In [12]:
# Pipeline para variables numéricas
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="mean"))
])

# Pipeline para variables categóricas
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# Preprocesador completo
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_cols),
        ("cat", categorical_transformer, cat_cols)
    ]
)

preprocessor

ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer', SimpleImputer())]),
                                 Index(['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous'], dtype='object')),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('onehot',
                                                  OneHotEncoder(handle_unknown='ignore'))]),
                                 Index(['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact',
       'month', 'poutcome'],
      dtype='object'))])